# MergeSortBottomUp (Java)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/MergeSortBottomUp_java.ipynb)

Capitolo 18 — Algoritmi ricorsivi e iterativi a confronto. Notebook eseguibile di `MergeSortBottomUp`: l'algoritmo in Java, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/18-ricorsivo-iterativo/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/MergeSortBottomUp_python.ipynb) e [C++](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/MergeSortBottomUp_cpp.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo_java.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. La prima cella verifica che il JDK ci sia e, se manca, lo installa. Ogni programma Java è scritto su un file dalla cella `%%writefile`, poi compilato con `javac` ed eseguito con `java` nella cella successiva.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `FibRec` → `fibRec` (camelCase).

## Preparazione

Su Colab il JDK di solito c'è già: la cella qui sotto lo verifica e, se manca, lo installa.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## MergeSort ricorsivo e MergeSortBottomUp

**MergeSortBottomUp**$(a,n)$ — *Input*: un array $a[1..n]$ di naturali. *Output*: l'array ordinato in modo non decrescente. La stessa suddivisione di Merge Sort, ma **senza ricorsione**: prima si fondono coppie di sotto-array di lunghezza $1$, poi di lunghezza $2$, $4$, … con la procedura `Merge` del capitolo 17 (qui ripetuta). Esegue $\lceil\log_2 n\rceil$ passate e $\Theta(n\log_2 n)$ passi, come la versione ricorsiva, e non usa la pila delle chiamate.

Prima l'esempio del capitolo $a=[5,2,4,6,1,3]$, passata per passata (gli indici sono da 0). Poi il confronto, su array pseudo-casuali (stesso generatore in tutti i linguaggi, seme $12345$, valori fra $1$ e $10^6$), del numero di **confronti** fra elementi di `InsertionSort`, di `MergeSort` e di `MergeSortBottomUp`: $\Theta(n^2)$ contro $\Theta(n\log_2 n)$, con le due versioni di Merge Sort praticamente uguali ("OK" indica che i tre algoritmi danno lo stesso array ordinato), e i tempi misurati.

In [2]:
%%writefile BottomUp.java
import java.util.*;

public class BottomUp {
    static long comparisons = 0;   // confronti fra elementi

    // merge(a, l, m, r) del capitolo 17
    static void merge(int[] a, int l, int m, int r) {
        int n1 = m - l + 1;
        int n2 = r - m;
        int[] L = new int[n1];
        int[] R = new int[n2];
        for (int i = 0; i < n1; i++) {
            L[i] = a[l + i];
        }
        for (int j = 0; j < n2; j++) {
            R[j] = a[m + 1 + j];
        }
        int i = 0, j = 0, k = l;
        while (i < n1 && j < n2) {
            comparisons++;
            if (L[i] <= R[j]) {
                a[k] = L[i];
                i++;
            } else {
                a[k] = R[j];
                j++;
            }
            k++;
        }
        while (i < n1) {
            a[k] = L[i];
            i++;
            k++;
        }
        while (j < n2) {
            a[k] = R[j];
            j++;
            k++;
        }
    }

    static void mergeSort(int[] a, int l, int r) {
        if (l < r) {
            int m = l + (r - l) / 2;
            mergeSort(a, l, m);
            mergeSort(a, m + 1, r);
            merge(a, l, m, r);
        }
    }

    // mergeSortBottomUp(a, n): ordina a[0..n-1] senza ricorsione
    static void mergeSortBottomUp(int[] a, int n) {
        int width = 1;
        while (width < n) {
            int l = 0;
            while (l + width < n) {                 // pseudocodice: l + width <= n
                int m = l + width - 1;
                int r = Math.min(l + 2 * width - 1, n - 1);
                merge(a, l, m, r);
                l = l + 2 * width;
            }
            width = 2 * width;
        }
    }

    static void mergeSortBottomUpTrace(int[] a, int n) {
        int width = 1;
        int passes = 0;
        while (width < n) {
            int l = 0;
            StringBuilder parts = new StringBuilder();
            while (l + width < n) {
                int m = l + width - 1;
                int r = Math.min(l + 2 * width - 1, n - 1);
                merge(a, l, m, r);
                if (parts.length() > 0) parts.append(", ");
                parts.append("a[" + l + ".." + m + "] con a[" + (m + 1) + ".." + r + "]");
                l = l + 2 * width;
            }
            passes++;
            System.out.println("width=" + width + ": merge " + parts + " => " + Arrays.toString(a));
            width = 2 * width;
        }
        System.out.println("passate = " + passes);
    }

    // insertionSort(a, n) (capitolo 3), con il conteggio dei confronti
    static void insertionSort(int[] a, int n) {
        for (int j = 1; j < n; j++) {
            int key = a[j];
            int i = j - 1;
            while (i >= 0) {
                comparisons++;
                if (a[i] > key) {
                    a[i + 1] = a[i];
                    i = i - 1;
                } else {
                    break;
                }
            }
            a[i + 1] = key;
        }
    }

    // Generatore congruenziale lineare: stessi numeri in Python, C++ e Java
    static int[] randomArray(int n, long seed) {
        long x = seed;
        int[] a = new int[n];
        for (int i = 0; i < n; i++) {
            x = (x * 1103515245L + 12345L) % 2147483648L;
            a[i] = 1 + (int) (x % 1000000L);
        }
        return a;
    }

    static double nowMs() {
        return System.nanoTime() / 1e6;
    }

    public static void main(String[] args) {
        int[] a = {5, 2, 4, 6, 1, 3};
        System.out.println("a = " + Arrays.toString(a) + "   n = " + 6);
        mergeSortBottomUpTrace(a, 6);

        System.out.println();
        System.out.printf("%6s %14s %10s %9s  confronti%n", "n", "InsertionSort", "MergeSort", "BottomUp");
        double[][] times = new double[3][];
        int idx = 0;
        for (int n : new int[]{1000, 2000, 4000}) {
            int[] base = randomArray(n, 12345);
            int[] x1 = base.clone(), x2 = base.clone(), x3 = base.clone();
            comparisons = 0;
            double t0 = nowMs(); insertionSort(x1, n); double t1 = nowMs() - t0;
            long c1 = comparisons;
            comparisons = 0;
            t0 = nowMs(); mergeSort(x2, 0, n - 1); double t2 = nowMs() - t0;
            long c2 = comparisons;
            comparisons = 0;
            t0 = nowMs(); mergeSortBottomUp(x3, n); double t3 = nowMs() - t0;
            long c3 = comparisons;
            int[] ord = base.clone();
            Arrays.sort(ord);
            String ok = (Arrays.equals(x1, x2) && Arrays.equals(x2, x3) && Arrays.equals(x3, ord)) ? "OK" : "ERROR";
            System.out.printf("%6d %14d %10d %9d  %s%n", n, c1, c2, c3, ok);
            times[idx++] = new double[]{n, t1, t2, t3};
        }
        System.out.println();
        System.out.println("Tempi misurati (dipendono dalla macchina):");
        for (double[] t : times) {
            System.out.printf("n=%-6d InsertionSort %10.3f ms   MergeSort %8.3f ms   MergeSortBottomUp %8.3f ms%n", (int) t[0], t[1], t[2], t[3]);
        }
    }
}

Writing BottomUp.java


In [3]:
!javac BottomUp.java && java BottomUp

a = [5, 2, 4, 6, 1, 3]   n = 6
width=1: merge a[0..0] con a[1..1], a[2..2] con a[3..3], a[4..4] con a[5..5] => [2, 5, 4, 6, 1, 3]
width=2: merge a[0..1] con a[2..3] => [2, 4, 5, 6, 1, 3]
width=4: merge a[0..3] con a[4..5] => [1, 2, 3, 4, 5, 6]
passate = 3

     n  InsertionSort  MergeSort  BottomUp  confronti
  1000         248782       8716      8740  OK
  2000        1010300      19415     19438  OK
  4000        4027558      42874     42876  OK

Tempi misurati (dipendono dalla macchina):
n=1000   InsertionSort     11.919 ms   MergeSort    1.116 ms   MergeSortBottomUp    1.064 ms
n=2000   InsertionSort      6.723 ms   MergeSort    4.060 ms   MergeSortBottomUp    0.488 ms
n=4000   InsertionSort      5.738 ms   MergeSort    0.980 ms   MergeSortBottomUp    1.203 ms


**Prova tu.** Cambia $n$ (con $n=8000$ i confronti di `InsertionSort` quadruplicano rispetto a $n=4000$, quelli di Merge Sort poco più del doppio).

In [4]:
%%writefile ProvaBottomUp.java
import java.util.*;

public class ProvaBottomUp {
    public static void main(String[] args) {
        // Prova con n = 8000: i confronti di InsertionSort quadruplicano rispetto a n = 4000, quelli di Merge Sort poco più del doppio.
        int n = 8000;
        int[] base = BottomUp.randomArray(n, 12345);
        int[] x1 = base.clone(), x2 = base.clone();
        BottomUp.comparisons = 0;
        BottomUp.insertionSort(x1, n);
        long c1 = BottomUp.comparisons;
        BottomUp.comparisons = 0;
        BottomUp.mergeSortBottomUp(x2, n);
        System.out.println(c1 + " " + BottomUp.comparisons + " " + (Arrays.equals(x1, x2) ? "True" : "False"));
    }
}

Writing ProvaBottomUp.java


In [5]:
!javac ProvaBottomUp.java && java ProvaBottomUp

16115652 93756 True


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).